# EvoVariant-TR post-hoc adaptation: judge demo

**Study:** `POSTHOC-FOUNDATION-ADAPTATION-001`  
**Protocol:** frozen, SHA-256 `07c93b4657e84a4ddfbdc2df1af0f467f80959e0534a67840b4bf2b2b04a2c2c`  
**Evidence label:** post-hoc development adaptation. Sections with no persisted result say `PENDING` or `NOT RUN`; no values are inferred from plans.

Run the source-display cells to inspect the implementation itself. Result cells read machine-readable artifacts and remain empty until a stage writes them. This notebook never loads the 946-row locked temporal test.


In [ ]:
from pathlib import Path
import ast, json, sys

ROOT = Path('/content/EvoVariant') if Path('/content/EvoVariant').exists() else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
DRIVE = Path('/content/drive/MyDrive/EvoVariantTR')
PROTOCOL = ROOT / 'research/adaptation/protocol.yaml'

def show_definition(relative_path, name):
    path = ROOT / relative_path
    source = path.read_text(encoding='utf-8')
    tree = ast.parse(source)
    matches = [node for node in ast.walk(tree) if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and node.name == name]
    if len(matches) != 1:
        raise RuntimeError(f'{relative_path}: expected one definition named {name}, found {len(matches)}')
    print(f'# {relative_path} :: {name}\n')
    print(ast.get_source_segment(source, matches[0]))

def show_context(relative_path, needle, before=3, after=18):
    lines = (ROOT / relative_path).read_text(encoding='utf-8').splitlines()
    indexes = [i for i, line in enumerate(lines) if needle in line]
    if not indexes:
        print(f'NOT FOUND: {needle}')
        return
    i = indexes[0]
    start, end = max(0, i-before), min(len(lines), i+after)
    print(f'# {relative_path}:{start+1}-{end}\n')
    print('\n'.join(lines[start:end]))


## 1. Project goal

Test whether task-specific adaptation of feasible genomic foundation models improves P/LP-vs-B/LB resolution-direction discrimination versus frozen representations on the formal gene-held-out 801-row adaptation holdout. This is a research-cohort comparison, not a clinical claim.


## 2. Frozen baseline

The original temporal baseline is a separate 946-row locked evaluation: AUROC `0.909225974`, bound to `main` commit `30b515314d5f8c8be7c83c96b1f576fb7225c869` and tag `evovariant-tr-baseline-v1` (`1003bc5a20973145e0e096ff7b0f3424045d07e6`). It is historical anchor evidence, not the adaptation holdout.


## 3. Why adaptation is separate

The adaptation cohort, labels, selection method, and evaluation stage differ from the frozen temporal baseline. The 946 locked IDs and labels are not loaded for adaptation selection, fitting, calibration, thresholding, or comparisons.


## 4. Dataset and splits

| Split | Rows | Negative | Positive | Role |
|---|---:|---:|---:|---|
| TRAIN | 3,199 | 2,645 | 554 | grouped CV, OOF calibration, final fit |
| VALIDATION | 801 | 581 | 220 | one-shot post-selection holdout |

TRAIN SHA-256: `32bf517ec8bc401d29f611e83a8c8c81eafc0d1f19886d2650a3bf441df044e1`  
VALIDATION SHA-256: `b31d884860fcf07b6f7f453c3ef148886913f381965318e9c1da341d1eaf3c8b`  
Development manifest SHA-256: `f4a9e53bd96c60dd9bd949568adb7a6bece3ff01bd4cceb76f71f1380e16e782`


## 5. Leakage controls

Three-fold `StratifiedGroupKFold` groups by gene inside TRAIN. The formal manifests are hash-pinned; TRAIN/VALIDATION have zero gene and normalized-ID overlap; adaptation IDs have zero overlap with the 946 locked manifest. Selection and calibration artifacts bind to the TRAIN manifest and protocol hash. The holdout evaluator requires a closed selection lock and refuses a second output.


## 6. Sequence construction

GRCh38 windows are centered at the variant, fixed to the configured context, and alternate sequence creation validates the REF allele before substitution. The live Colab check matched all 4,000 TRAIN/VALIDATION REF alleles to the verified reference FASTA (SHA-256 `5be01555d98347fdb3714dc84c6f77c9d8bc774adcf32c6f7a8fa06f5baf5e51`).

In [ ]:
show_definition('src/evovariant_tr/adaptation/data.py', 'reference_window')
show_definition('src/evovariant_tr/adaptation/data.py', 'paired_sequences')
show_definition('src/evovariant_tr/adaptation/data.py', 'verify_reference_rows')


## 7. Caduceus architecture

Pinned model: `kuleshov-group/caduceus-ph_seqlen-131k_d_model-256_n_layer-16`, revision `b0477522ac5d044ad03578aa724ec8e4bdbd405b`. A shared encoder produces masked mean-pooled sequence embeddings; the classification head is small and explicit.

## 8. Paired reference/alternate design

The shared encoder processes ref and alt sequences. The head receives `z_ref`, `z_alt`, `z_alt-z_ref`, and `abs(z_alt-z_ref)`. Forward and reverse-complement ref/alt logits are averaged by a predeclared rule.

In [ ]:
show_definition('src/evovariant_tr/adaptation/models.py', '_last_hidden')
show_definition('src/evovariant_tr/adaptation/models.py', '_mean_pool')
show_definition('src/evovariant_tr/adaptation/models.py', 'pair_logits')
show_definition('src/evovariant_tr/adaptation/models.py', 'forward')


## 9. Actual fine-tuning code

The next cell prints the exact training loop used by the CLI, not a paraphrase or notebook-only rewrite.

In [ ]:
show_definition('src/evovariant_tr/adaptation/training.py', 'train_epoch')


## 10. Loss

`BCEWithLogitsLoss(pos_weight = fold/train negatives ÷ fold/train positives)` is computed from the current training rows only. HPO computes this separately inside each fold; no VALIDATION labels contribute to class weights.

In [ ]:
show_context('src/evovariant_tr/adaptation/training.py', 'pos_weight = torch.tensor', before=4, after=12)


## 11. Optimizer

The runner uses AdamW on trainable parameters only. Learning rate and weight decay are fixed for the initial frozen-head baseline and searched inside the predeclared TRAIN-only HPO space later.

In [ ]:
show_context('scripts/adaptation/train_caduceus.py', 'optimizer = torch.optim.AdamW', before=5, after=5)


## 12. Scheduler

No learning-rate scheduler is used in the current implementation. Each trial uses a fixed AdamW learning rate, which is itself a searched parameter. This is the implementation status; no scheduler curve is claimed.

## 13. Forward pass

The model's actual shared-encoder path and orientation aggregation are shown below.

In [ ]:
show_definition('src/evovariant_tr/adaptation/models.py', 'encode')
show_definition('src/evovariant_tr/adaptation/models.py', 'pair_logits')
show_definition('src/evovariant_tr/adaptation/models.py', 'forward')


## 14. Backpropagation

For each microbatch the implementation divides loss by the accumulation group size, then backpropagates through the paired logits.

In [ ]:
show_context('src/evovariant_tr/adaptation/training.py', 'scaler.scale(loss / group_size).backward()', before=9, after=12)


## 15. Gradient clipping

Before each optimizer step, gradients are unscaled and clipped to norm 1.0.

In [ ]:
show_context('src/evovariant_tr/adaptation/training.py', 'clip_grad_norm_', before=3, after=5)


## 16. Mixed precision

CUDA training enables autocast and a gradient scaler. The observed T4 runtime is CUDA 12.1 with PyTorch `2.2.0+cu121`; the isolated environment pins NumPy `1.26.4` for binary compatibility.

In [ ]:
show_definition('src/evovariant_tr/adaptation/training.py', '_autocast')
show_context('src/evovariant_tr/adaptation/training.py', 'GradScaler', before=3, after=3)


## 17. Early stopping

TRAIN-fold validation AUROC controls HPO early stopping only, with patience 2 and `min_delta=0.0001`; the 801 holdout is not used. The final all-TRAIN run uses an epoch count derived from the median fold best epoch after HPO selection closes.

In [ ]:
show_context('scripts/adaptation/run_caduceus_hpo.py', 'for epoch in range(start_epoch, epoch_cap):', before=3, after=32)


## 18. Checkpointing and resume

Checkpoints are atomically replaced and include model, optimizer, RNG state, epoch, metrics, protocol hash, and config/data metadata. HPO stores per-trial/per-fold checkpoints on Drive and reopens its SQLite study.

In [ ]:
show_definition('src/evovariant_tr/adaptation/checkpointing.py', 'save_checkpoint')
show_definition('src/evovariant_tr/adaptation/checkpointing.py', 'load_checkpoint')


## 19. Hyperparameter optimization

The frozen protocol predeclares 8–12 trials, 3 gene-grouped folds, pruning, and the search space. Selection prioritizes mean fold AUROC, then AUPRC, MCC, Brier, runtime, and trainable parameter count. The actual nested objective follows.

In [ ]:
show_definition('scripts/adaptation/run_caduceus_hpo.py', 'objective')


## 20. Caduceus results

Status is read from Drive artifacts below. The initial T4 job is TRAIN-only, frozen encoder/head training, 3 epochs, seed 42, 8,192 bp. A smoke test passed at 128 bp; it does not establish full-cohort training completion.

In [ ]:
from pathlib import Path
for path in [
    DRIVE/'runs/caduceus_smoke.json',
    DRIVE/'checkpoints/caduceus_frozen_head/run.json',
    DRIVE/'hpo/caduceus_hpo.json',
    DRIVE/'hpo/selection_closed.json',
]:
    print(f'--- {path} ---')
    print(path.read_text() if path.exists() else 'PENDING')


## 21. Nucleotide Transformer v2 PEFT

Target revision is pinned in the model registry. The Caduceus track must be persisted before this track begins. As of this notebook's current code, NT frozen/PEFT training scripts do not exist and no NT results are claimed; this section stays `NOT RUN` until a real stage artifact exists.

In [ ]:
show_definition('src/evovariant_tr/adaptation/models.py', 'load_backbone')
for path in [DRIVE/'runs/nt_frozen/run.json', DRIVE/'runs/nt_peft/run.json']:
    print(path, 'PRESENT' if path.exists() else 'NOT RUN')


## 22. Calibration

The checked-in adaptation helper currently implements temperature scaling fitted on TRAIN OOF logits. The frozen protocol also requests Platt and isotonic comparison; these are not represented by this helper yet, so calibration is incomplete until those candidates and the final frozen calibration are persisted.

In [ ]:
show_definition('src/evovariant_tr/adaptation/calibration.py', 'fit_temperature')
show_definition('src/evovariant_tr/adaptation/calibration.py', 'apply_temperature')


## 23. Abstention

The present helper selects an MCC threshold on TRAIN OOF predictions. Protocol coverage targets (0.5, 0.75, 0.9, 1.0), risk-coverage, and holdout selective metrics remain pending.

In [ ]:
show_definition('src/evovariant_tr/adaptation/calibration.py', 'select_abstention_threshold')


## 24. Ensembles

Candidate comparisons are frozen Evo2 plus adapted Caduceus, and optionally NT PEFT. Weights must be fitted on TRAIN OOF predictions; holdout weights are prohibited. Adaptation-specific ensemble code and prediction artifacts are pending.

## 25. Ablations

Predeclared representation, orientation, fine-tuning, calibration, and ensemble ablations will be reported only when they have measured TRAIN-CV or permitted holdout artifacts. No trend is inferred from missing runs.

## 26. Learning curves

Fractions are fixed at 10%, 25%, 50%, 75%, and 100%, using deterministic grouped/class-aware TRAIN subsets. No curve is reported until all supported points are measured.

## 27. Context-length trends

Predeclared contexts: 512, 1,024, 2,048, 4,096, and 8,192 bp. The active baseline uses 8,192 bp. Missing context points remain missing; there is no interpolation.

## 28. Metrics and formulas

Accuracy = (TP+TN)/N; precision = TP/(TP+FP); recall = TP/(TP+FN); specificity = TN/(TN+FP); F1 = 2PR/(P+R); balanced accuracy = (recall+specificity)/2. AUROC ranks positives against negatives; AUPRC summarizes precision over recall; MCC summarizes all four confusion cells; Brier is mean squared probability error; NLL is binary log loss; ECE is binned calibration gap. Probability MAE is supplementary, not a primary endpoint.

In [ ]:
show_definition('src/evovariant_tr/adaptation/metrics.py', 'binary_metrics')


## 29. ROC and precision-recall

`PENDING`: curves require a permitted, persisted prediction CSV. This notebook does not draw synthetic curves.

## 30. Confusion matrices

`PENDING`: confusion cells will be computed from the one-shot holdout prediction artifact after selection is frozen.

## 31. Calibration and reliability

`PENDING`: reliability plots require measured OOF-fitted calibrators and one-shot holdout probabilities. Calibration cannot be fitted on the holdout.

## 32. Runtime and VRAM

The smoke stage measured 10.73 s and peak allocation 452,699,136 bytes at 128 bp. Full 8,192 bp training and holdout resource metrics are read only from completed artifacts.

In [ ]:
for path in [DRIVE/'runs/caduceus_smoke.json', DRIVE/'checkpoints/caduceus_frozen_head/run.json']:
    print(f'--- {path} ---')
    print(path.read_text() if path.exists() else 'PENDING')


## 33. Error analysis

Errors by chromosome/gene, confidence, and model disagreement require holdout predictions after finalization; every subgroup will show sample size and tiny groups will be suppressed. No retuning follows this analysis.

## 34. Contributions

The adaptation branch adds a pinned Caduceus paired ref/alt head, reverse-complement averaging, TRAIN-only gene-grouped HPO, atomic checkpoints, and leakage checks. Final empirical contribution claims depend on completed evidence, not on implementation alone.

## 35. Limitations

The development sample is class-stratified, so prevalence-sensitive metrics describe this study distribution. The historical 946-row temporal baseline and 801-row adaptation holdout are not interchangeable. This work does not establish clinical validity, causality, treatment utility, or universal genomic performance. T4 time may constrain HPO, full fine-tuning, NT PEFT, and robustness breadth.

## 36. Reproducibility

Protocol hash, split hashes, pinned model revisions, environment, checkpoint hashes, and stage state are persisted under `MyDrive/EvoVariantTR`. Reconnect by reading `state/adaptation_state.json` and resuming the first incomplete stage; do not rerun completed stages or run all historical cells.